# C1 · CLIP zero-shot vs linear probe on CIFAR-10 (12 min)

**Question:** how good is classification with *no* training (just class names as prompts), and how many labels does a simple classifier on the same CLIP features need to beat it?

Flow: embed images once → zero-shot with one prompt → prompt ensemble → per-class errors → few-shot linear probe curve → error gallery.
Outputs go to `outputs/c1/`.

In [ ]:
# Make the project folder importable (works from notebooks/ or the project root)
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

In [ ]:
# ---- configuration -------------------------------------------------------
CLIP_ID   = "openai/clip-vit-base-patch32"   # try "openai/clip-vit-large-patch14"
N_TEST    = 2000                              # test images to score (10,000 max)
POOL_PER_CLASS = 100                          # labelled train images per class available to the probe
SHOTS     = [1, 2, 5, 10, 25, 100]            # labels per class for the probe curve
SEEDS     = [0, 1, 2]
TEMPLATES = [
    "a photo of a {}.",
    "a blurry photo of a {}.",
    "a low resolution photo of a {}.",
    "a photo of the small {}.",
    "a close-up photo of a {}.",
    "a pixelated photo of a {}.",
    "an image of a {}.",
]

In [ ]:
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from vlmkit.common import get_device, out_dir, save_json, show_grid, Timer
from vlmkit.clip_utils import Clip, accuracy
from vlmkit import data

DEVICE, OUT = get_device(), out_dir("c1")
test, train = data.cifar10(train=False), data.cifar10(train=True)
CLASSES = test.classes
rng = np.random.default_rng(0)
test_idx = np.arange(min(N_TEST, len(test)))
y_test = np.array([test.targets[i] for i in test_idx])
train_targets = np.array(train.targets)
pool_idx = np.concatenate([rng.choice(np.where(train_targets == c)[0], POOL_PER_CLASS, replace=False)
                           for c in range(len(CLASSES))])
y_pool = train_targets[pool_idx]
print(f"device={DEVICE}  test={len(test_idx)}  probe pool={len(pool_idx)}")

## 1. Embed images once (cached)

In [ ]:
clip = Clip(CLIP_ID, DEVICE)
cache = OUT / f"emb_{CLIP_ID.replace('/', '_')}_{len(test_idx)}_{len(pool_idx)}.npz"
if cache.exists():
    z = np.load(cache); V_test, V_pool = torch.from_numpy(z["test"]), torch.from_numpy(z["pool"])
else:
    with Timer() as t:
        V_test = clip.images([test[i][0] for i in test_idx], progress=True)
        V_pool = clip.images([train[i][0] for i in pool_idx], progress=True)
    print(f"embedded {len(test_idx) + len(pool_idx)} images in {t.seconds:.1f} s")
    np.savez(cache, test=V_test.numpy(), pool=V_pool.numpy())
V_test.shape, V_pool.shape

In [ ]:
# What the embeddings look like: 2-D PCA map of the test images, coloured by true class.
# Classes CLIP finds easy form tight islands; overlapping clouds (cat/dog, automobile/truck) predict confusions.
from sklearn.decomposition import PCA

P = PCA(2, random_state=0).fit_transform(V_test.numpy())
fig, ax = plt.subplots(figsize=(7, 5.5))
for k, c in enumerate(CLASSES):
    m = y_test == k
    ax.scatter(P[m, 0], P[m, 1], s=6, alpha=0.6, label=c)
    ax.annotate(c, P[m].mean(0), fontsize=9, weight="bold", ha="center")
ax.set_xticks([]); ax.set_yticks([]); ax.legend(markerscale=3, fontsize=8, loc="best")
ax.set_title("CLIP image embeddings, PCA to 2-D"); plt.tight_layout()
plt.savefig(OUT / "embedding_map.png", dpi=120); plt.show()

## 2. Zero-shot with a single prompt

In [ ]:
T_single = clip.class_embeddings(CLASSES, ["a photo of a {}."])
pred_single = (V_test @ T_single.T).argmax(1).numpy()
acc_single = accuracy(pred_single, y_test)
print(f"zero-shot, one prompt: {acc_single:.3f}")

## 3. Prompt sensitivity and ensembling

Each template alone, then all templates averaged per class.

In [ ]:
rows = []
for tpl in TEMPLATES:
    T = clip.class_embeddings(CLASSES, [tpl])
    rows.append({"template": tpl, "accuracy": accuracy((V_test @ T.T).argmax(1).numpy(), y_test)})
T_ens = clip.class_embeddings(CLASSES, TEMPLATES)
pred_zs = (V_test @ T_ens.T).argmax(1).numpy()
acc_ens = accuracy(pred_zs, y_test)
rows.append({"template": "ENSEMBLE (all above)", "accuracy": acc_ens})
prompt_df = pd.DataFrame(rows).sort_values("accuracy")
prompt_df.to_csv(OUT / "prompt_sensitivity.csv", index=False)

ax = prompt_df.plot.barh(x="template", y="accuracy", legend=False, figsize=(7, 3.5), color="#4C78A8")
ax.set_xlim(prompt_df.accuracy.min() - 0.05, 1); ax.set_xlabel("zero-shot accuracy")
ax.set_title("Prompt wording changes zero-shot accuracy"); plt.tight_layout()
plt.savefig(OUT / "prompt_sensitivity.png", dpi=120); plt.show()
print(f"spread across single prompts: {prompt_df.accuracy.iloc[:-1].max() - prompt_df.accuracy.iloc[:-1].min():.3f}")

## 4. Where does zero-shot fail? Per-class accuracy and confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

per_class = pd.Series({c: accuracy(pred_zs[y_test == k], y_test[y_test == k]) for k, c in enumerate(CLASSES)})
print(per_class.sort_values().round(3).to_string())
fig, ax = plt.subplots(figsize=(6.5, 6))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred_zs, normalize="true"), display_labels=CLASSES).plot(
    ax=ax, cmap="Blues", values_format=".2f", colorbar=False, xticks_rotation=45)
ax.set_title("Zero-shot confusion (rows = true class)"); plt.tight_layout()
plt.savefig(OUT / "confusion_zero_shot.png", dpi=120); plt.show()

## 5. Linear probe: how many labels beat zero-shot?

Logistic regression on the same frozen CLIP features, k labelled images per class, averaged over 3 random draws.

In [ ]:
from sklearn.linear_model import LogisticRegression

Xp, Xt = V_pool.numpy(), V_test.numpy()
curve = []
for k in SHOTS:
    accs = []
    for seed in SEEDS:
        r = np.random.default_rng(seed)
        sel = np.concatenate([r.choice(np.where(y_pool == c)[0], k, replace=False) for c in range(len(CLASSES))])
        clf = LogisticRegression(max_iter=3000, C=10.0).fit(Xp[sel], y_pool[sel])
        accs.append(clf.score(Xt, y_test))
    curve.append({"shots_per_class": k, "accuracy_mean": np.mean(accs), "accuracy_std": np.std(accs)})
curve_df = pd.DataFrame(curve)
curve_df.to_csv(OUT / "probe_curve.csv", index=False)

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.errorbar(curve_df.shots_per_class, curve_df.accuracy_mean, yerr=curve_df.accuracy_std,
            marker="o", capsize=3, label="linear probe (CLIP features)")
ax.axhline(acc_ens, color="#E45756", ls="--", label=f"zero-shot ensemble ({acc_ens:.3f})")
ax.set_xscale("log"); ax.set_xticks(SHOTS); ax.set_xticklabels(SHOTS)
ax.set_xlabel("labelled images per class (log scale)"); ax.set_ylabel("test accuracy")
crossing = curve_df[curve_df.accuracy_mean > acc_ens].shots_per_class.min()
ax.set_title(f"Probe overtakes zero-shot at ~{crossing} labels per class" if pd.notna(crossing)
             else "Zero-shot stays ahead at every label budget tried")
ax.legend(); plt.tight_layout(); plt.savefig(OUT / "probe_vs_zero_shot.png", dpi=120); plt.show()
curve_df.round(3)

## 6. Error gallery

In [ ]:
wrong = np.where(pred_zs != y_test)[0][:10]
show_grid([test[int(test_idx[i])][0].resize((128, 128)) for i in wrong],
          [f"true {CLASSES[y_test[i]]}\npred {CLASSES[pred_zs[i]]}" for i in wrong],
          cols=5, size=2.2, save=OUT / "errors_zero_shot.png")

In [ ]:
summary = {
    "clip": CLIP_ID, "n_test": int(len(test_idx)),
    "zero_shot_single_prompt": acc_single, "zero_shot_ensemble": acc_ens,
    "best_single_template": prompt_df.iloc[:-1].sort_values("accuracy").iloc[-1].to_dict(),
    "worst_class": per_class.idxmin(), "worst_class_accuracy": float(per_class.min()),
    "probe_curve": curve_df.round(4).to_dict("records"),
    "probe_beats_zero_shot_at_shots": None if pd.isna(crossing) else int(crossing),
}
save_json(summary, OUT / "summary.json")
summary

## 7. Try it live: your own labels on any image

Pick a CIFAR test image with the slider (or paste an image URL), type any comma-separated labels and press **Run**.
No training happens: the labels you type *are* the classifier.

In [ ]:
import io, urllib.request
from PIL import Image
from vlmkit.common import live

def zero_shot_panel(image_index, labels, image_url):
    if image_url.strip():
        req = urllib.request.Request(image_url.strip(), headers={"User-Agent": "open-data-c-vlm"})
        im, true = Image.open(io.BytesIO(urllib.request.urlopen(req).read())).convert("RGB"), "?"
    else:
        im, y = test[image_index]
        true = CLASSES[y]
    names = [s.strip() for s in labels.split(",") if s.strip()]
    T = clip.class_embeddings(names, TEMPLATES)
    probs = (100 * clip.images([im]) @ T.T).softmax(-1)[0].numpy()       # CLIP's logit scale is 100
    order = np.argsort(probs)[-12:]
    fig, (a, b) = plt.subplots(1, 2, figsize=(9, 3.4), gridspec_kw={"width_ratios": [1, 2]})
    a.imshow(im.resize((224, 224))); a.axis("off"); a.set_title(f"true: {true}")
    b.barh([names[i] for i in order], probs[order], color="#4C78A8"); b.set_xlim(0, 1)
    b.set_xlabel("zero-shot probability"); b.set_title(f"prediction: {names[order[-1]]}")
    plt.tight_layout(); plt.show()

live(zero_shot_panel, image_index=(0, len(test) - 1, 1, 3),
     labels=", ".join(CLASSES) + ", tractor, toy car, cartoon cat",
     image_url="")

## Exercises for students

1. Add your own templates to `TEMPLATES` (for example "a photo of a {}, a type of vehicle.") and re-run sections 3–5.
2. Switch `CLIP_ID` to `openai/clip-vit-large-patch14`: how much do zero-shot and the probe gain, and at what cost in embedding time?
3. **Transfer to your project:** replace CIFAR classes with your own scene labels (day/night, rain, crash/no crash) and measure zero-shot on 50 of your frames.